# Calculate TCGI

This notebook computes the TCGI for a given model and time period. It should be run **after** completing the following preprocessing steps:

- `./calculate_avort_ws.py`: calculate absolute vorticity (`avort`) and vertical wind shear (`ws`).
- `./calculate_swv.py`: calculate saturated water vapor (`swv`).
- `./tcpyPI`: calculate tropical cyclone potential intensity (`PI`).
- `./cal_TCGI_v2_part1.ipynb`: calculate model climatology fields.

In addition to the four variables above, this notebook requires:
- Total column water vapor (precipitable water, `prw`) — model output in units of kg m⁻².
- Land mask — model land mask at the original model resolution, where ocean grid points have a value of zero.
- ERA5 climatologies (1981–2010) - Located in `./ERA5_clim/`.
- ERA5 variable min/max files - Located in `./ERA5_minmax/`.

---
### Workflow Overview

1. **Step 1** — Regrid all variables to 2° × 2° and apply the ERA5 land mask.
2. **Step 2** — Compute mean-state bias correction terms (model climatology minus ERA5 climatology both over 1981-2010).
3. **Step 3** — Apply the bias correction and replace any unphysical values.
4. **Step 4** — Calculate `TCGI-CRH` and `TCGI-SD`.

---
### References
- Lee, Chia-Ying, et al. "Statistical–dynamical downscaling projections of tropical cyclone activity in a warming climate: Two diverging genesis scenarios." Journal of Climate 33.11 (2020): 4815-4834.
- Tippett, Michael K., Suzana J. Camargo, and Adam H. Sobel. "A Poisson regression index for tropical cyclone genesis and the role of large-scale vorticity in genesis." Journal of Climate 24.9 (2011): 2335-2357.
- Fosu, Boniface O., et al. "Assessing future tropical cyclone risk using downscaled CMIP6 projections." Journal of Catastrophe Risk and Resilience 2024.2 (2024): 1.
- Camargo, Suzana J., et al. "Testing the performance of tropical cyclone genesis indices in future climates using the HiRAM model." Journal of Climate 27.24 (2014): 9171-9196.

In [3]:
import xarray as xr
import numpy as np
import xesmf as xe
import os

In [ ]:
def replace_land(file_path, input_var, model_lmask, prw_varname):
    """
    Fill land grid points with the zonal mean climatology at each latitude.

    This pre-processing step prevents artefacts in coastal grid points that
    can arise during regridding when land points carry fill values.
    """
    disk_var = "vmax" if input_var == "PI" else input_var
    disk_var = prw_varname if input_var == "prw" else disk_var
    with xr.open_dataset(file_path) as dataset:
        ocean_mask = (model_lmask == 0).data
        
        # Compute the time- and zonally-averaged climatology over ocean only
        ocean_only = dataset[disk_var].where(ocean_mask)
        zonal_clim = ocean_only.mean('time').mean(dim='lon', skipna=True)

        # Broadcast the 1-D zonal mean back to the full (lat, lon) grid
        lat_mean = zonal_clim.expand_dims(dim={'lon': dataset['lon'].data}, axis=-1)

        # Replace land points with the zonal mean; keep ocean points unchanged
        ds_filled = dataset[disk_var].where(ocean_mask, other=lat_mean)

        # Remove the 'type' coordinate added by some ERA5 downloads
        if 'type' in ds_filled.coords:
            ds_filled = ds_filled.drop_vars('type')
    return ds_filled

## Configuration

Edit the **User Settings** block below. Do **not** modify the Default Settings block.

In [ ]:
# ====================================================================
# USER SETTINGS — modify these for your dataset
# ====================================================================

# Directory containing the preprocessed variable files (avort, swv, PI, ws)
work_dir = 'WORK_DIRECTORY'

# Directory containing the model precipitable water (prw) output
prw_dir = 'PRW_DIRECTORY'

# File name template for the prw files. Use 'YYYY' as a placeholder for the
# year, e.g. 'prw_YYYY.nc' will be resolved to 'prw_1981.nc', 'prw_1982.nc', ...
prw_filename = 'prw_YYYY.nc'

# Variable name for precipitable water inside the prw NetCDF file (unit: kg m⁻²)
prw_varname = 'PRW_VARNAME'

# Land mask at the model's native resolution.
# The mask must contain a 2-D variable where ocean = 0.
# Replace 'lmask' below with the actual variable name in your file.
model_lmask = xr.open_dataset('MODEL_LANDMASK_FILE.nc')['lmask']

# Directory containing the 1981–2010 model climatology files produced by
# cal_TCGI_v2_part1.ipynb. Use your PARENT_WORK_DIR for multi-ensemble 
# mean climatology, or WORK_DIR for a single ensemble member.
clim_model_dir = 'MODEL_CLIM_DIR'

# Directory containing the ERA5 1981–2010 monthly climatology files (2° × 2°)
clim_era5_dir = './ERA5_clim/'

# Directory containing the ERA5 variable minimum/maximum files (2° × 2°)
minmax_dir = './ERA5_minmax/'

# First and last year for which TCGI will be calculated
cal_start, cal_end = 1951, 2100


# ====================================================================
# DEFAULT SETTINGS — do not modify
# ====================================================================

# Input variable keys and their corresponding output variable names
input_vars  = ['avort', 'swv', 'prw', 'PI', 'ws']
output_vars = ['avort', 'crh', 'sd',  'PI', 'ws']

# ERA5 land mask regridded to 2° × 2° (ocean = 0)
lmask_ERA5_2deg = xr.open_dataset('./ERA5_landmask_2deg.nc')['lmask']

# Target grid: 2° × 2° global grid
resolution = 2
ds_out = xr.Dataset({
    'lat': (['lat'], np.arange(-90, 90 + resolution, resolution)),
    'lon': (['lon'], np.arange(0, 360, resolution))
})


## Step 1 — Regrid to 2° × 2° and Apply Land Mask

For each year between `cal_start` and `cal_end`, this step:
1. Fills land points in `PI`, `swv`, and `prw` with the zonal-mean climatology at each latitude before regridding. This avoids spurious values in coastal grid cells that can arise from interpolation when neighbouring land points carry missing values.
2. Regrids all variables to a 2° × 2° global grid using bilinear interpolation.
3. Masks residual land points using the ERA5 2° land mask.
4. Derives the column relative humidity (`crh = prw / swv`) and saturation deficit (`sd = prw − swv`).
5. Saves each variable to `{work_dir}/2deg/{var}_{year}_2deg.nc`.

Years for which all 2° output files already exist (e.g., from Part 1) are skipped automatically.

In [ ]:
os.makedirs(f"{work_dir}/2deg", exist_ok=True)

# Variable metadata written to each output file
attributes = {
    'avort': {'units': 's-1',    'long_name': 'Absolute vorticity at 850 hPa'},
    'crh':   {'units': '1',      'long_name': 'Column relative humidity (prw / swv)'},
    'sd':    {'units': 'kg m-2', 'long_name': 'Saturation deficit (prw - swv)'},
    'PI':    {'units': 'm s-1',  'long_name': 'Potential intensity (vmax)'},
    'ws':    {'units': 'm s-1',  'long_name': 'Vertical wind shear (between 850 and 200 hPa)'},
}

print('### Step 1: Regridding variables to 2° × 2° and applying land mask...')

for iyear in range(cal_start, cal_end + 1):

    # Skip this year if all output files already exist
    all_exist = all(
        os.path.exists(f"{work_dir}/2deg/{var}_{iyear}_2deg.nc")
        for var in output_vars
    )
    if all_exist:
        print(f'  Skipping {iyear} (files already exist)')
        continue

    ds = {}   # accumulate regridded DataArrays for this year
    for input_var in input_vars:
        # Resolve the file path for this variable and year
        if input_var == 'prw':
            file_path = f"{prw_dir}/{prw_filename.replace('YYYY', str(iyear))}"
        else:
            file_path = f"{work_dir}/{input_var}_{iyear}.nc"

        # Fill land points with zonal mean for variables sensitive to coastal artefacts
        if input_var in ['swv', 'prw', 'PI']:
            ds_raw = replace_land(file_path, input_var, model_lmask, prw_varname)
        else:
            ds_raw = xr.open_dataset(file_path)[input_var]

        # Regrid to the 2° × 2° target grid
        regridder = xe.Regridder(ds_raw, ds_out, 'bilinear', periodic=True)
        dsr = regridder(ds_raw)

        # Replace spurious large values introduced by regridding with NaN before applying the land mask.
        # Note that the regridding can produce unphysically large values at coastal grid points. 
        # This step may not work perfectly, please check the output files for any remaining artefacts.
        dsr = dsr.where(np.abs(dsr) < 1e10)

        # Mask land points on the 2° grid using the ERA5 land mask
        ds[input_var] = dsr.where(lmask_ERA5_2deg == 0, other=np.nan)

    # Derive TCGI variables from the regridded fields
    computed = {
        'avort': ds['avort'],
        'crh':   ds['prw'] / ds['swv'],
        'sd':    ds['prw'] - ds['swv'],
        'PI':    ds['PI'],
        'ws':    ds['ws'],
    }

    # Attach metadata and save each output variable
    for var in output_vars:
        da = computed[var]
        da.attrs.update(attributes[var])
        da.rename(var).to_netcdf(
            f"{work_dir}/2deg/{var}_{iyear}_2deg.nc",
            encoding={var: {'zlib': True, 'complevel': 5}}, mode='w'
        )

    print(f'  Saved 2° files for {iyear}')

print('### Step 1 complete.')

## Step 2 — Compute the Mean-State Bias Correction Term

For each output variable, the bias correction term is defined as:

$$\Delta X = X_{\mathrm{ERA5\ clim}} - X_{\mathrm{model\ clim}}$$

where both climatologies cover the 1981–2010 baseline period and are on the 2° × 2° grid. The resulting correction has 12 time steps (one per month) and will be added to the model fields month by month in Step 3.

**Expected file names:**
- ERA5 climatology: `{clim_era5_dir}/{var}_ERA5_clim_1981-2010_2deg.nc`
- Model climatology: `{clim_model_dir}/{var}_1981-2010_clim_2deg.nc` *(output of Part 1)*

In [ ]:
clim_era5  = {}
clim_model = {}
cor_model  = {}

print('### Step 2: Computing bias correction terms...')

for var in output_vars:
    clim_era5[var]  = xr.open_dataset(f"{clim_era5_dir}{var}_ERA5_clim_1981-2010_2deg.nc")[var]
    clim_model[var] = xr.open_dataset(f"{clim_model_dir}/{var}_1981-2010_clim_2deg.nc")[var]
    cor_model[var]  = clim_era5[var] - clim_model[var]
    print(f'  Computed correction term for {var}')

print('### Step 2 complete.')

## Step 3 — Apply Bias Correction and Replace Unphysical Values

This step applies the monthly bias correction terms computed in Step 2 to each year of model data, then replaces any values that fall outside physically meaningful bounds.

The post-correction adjustments are:

- `avort`: Take absolute value, convert to 10⁻⁵ s⁻¹; clip at 3.7 × 10⁻⁵ s⁻¹ to improves sensitivity to near-equatorial meridional gradients while preventing over-prediction at higher latitudes (see Tippett et al. 2011).
- `crh`: Replace negative values with ERA5 monthly minimum; convert to %.
- `sd`: Replace positive values with ERA5 monthly maximum.
- `PI`: Replace negative values with ERA5 monthly minimum.
- `ws`: Replace negative values with ERA5 monthly minimum.

Bias corrected terms are saved to `{work_dir}/unbiased/{var}_{year}_2deg_unbiased.nc`.

In [ ]:
def add_climatology(month_group, climatology):
    # Extract the month number from the group
    month_num = int(month_group.time.dt.month.values[0])
    
    # Select the climatology data for the corresponding month
    climatology_month = climatology.sel(month=month_num)
    
    # Add the climatology to the month_group
    month_group, climatology_month = xr.align(month_group, climatology_month)
    return month_group + climatology_month

In [ ]:
# Load ERA5 variable bounds used to constrain unphysical post-correction values
ws_min  = xr.open_dataset(f"{minmax_dir}ws_min_ERA5_1981-2010_2deg.nc")['ws_min']
PI_min  = xr.open_dataset(f"{minmax_dir}PI_min_ERA5_1981-2010_2deg.nc")['PI_min']
sd_max  = xr.open_dataset(f"{minmax_dir}sd_max_ERA5_1981-2010_2deg.nc")['sd_max']
crh_min = xr.open_dataset(f"{minmax_dir}crh_min_ERA5_1981-2010_2deg.nc")['crh_min']

In [ ]:
os.makedirs(f"{work_dir}/unbiased", exist_ok=True)

print('### Step 3: Applying bias correction and replacing unphysical values...')

for iyear in range(cal_start, cal_end + 1):
    bc_ds = {}  # bias-corrected fields
    bcp_ds = {} # bias-corrected and physically constrained fields
    for var in output_vars:
        ori_ds = xr.open_dataset(f"{work_dir}/2deg/{var}_{iyear}_2deg.nc")[var]

        # Add the monthly correction term via groupby to ensure correct month alignment
        bc_ds[var] = ori_ds.groupby('time.month').apply(
            add_climatology, climatology=cor_model[var]
        )

        # Drop auxiliary coordinates that may cause issues in downstream operations
        coords_to_drop = [c for c in ['month', 'plev'] if c in bc_ds[var].coords]
        if coords_to_drop:
            bc_ds[var] = bc_ds[var].drop_vars(coords_to_drop)

    print(f'  Bias correction applied for {iyear}')

    # Replace unphysical post-correction values
    bcp_ds['avort'] = abs(bc_ds['avort']*1e+5).clip(min=None, max=3.7)
    bcp_ds['crh'] = xr.where(bc_ds['crh'] < 0, crh_min.sel(month=bc_ds["crh"].time.dt.month), bc_ds['crh'])*100
    bcp_ds['sd'] = xr.where(bc_ds['sd'] > 0, sd_max.sel(month=bc_ds["sd"].time.dt.month), bc_ds['sd'])
    bcp_ds['PI'] = xr.where(bc_ds['PI'] < 0, PI_min.sel(month=bc_ds["PI"].time.dt.month), bc_ds['PI'])
    bcp_ds['ws'] = xr.where(bc_ds['ws'] < 0, ws_min.sel(month=bc_ds["ws"].time.dt.month), bc_ds['ws'])

    # Save bias-corrected and physically constrained fields
    for var in output_vars:
        bcp_ds[var].rename(var).to_netcdf(
            f"{work_dir}/unbiased/{var}_{iyear}_2deg_unbiased.nc",
            encoding={var: {'zlib': True, 'complevel': 5}}, mode='w'
        )

print('### Step 3 complete.')

## Step 4 — Calculate TCGI

Two variants of TCGI are computed:
- TCGI-CRH (`mu_crh`): uses column relative humidity as the moisture term.
- TCGI-SD (`mu_sd`): uses saturation deficit as the moisture term.

Both follow the Poisson regression form (Camargo et al. 2014):

$$\mu = \exp\left(b_0 + b_\eta \cdot \eta + b_H \cdot H + b_T \cdot T + b_V \cdot V + \log(\cos\phi \cdot \Delta x \cdot \Delta y)\right)$$

where $\eta$ is absolute vorticity, $H$ is the humidity term (CRH or SD), $T$ is potential intensity, $V$ is vertical wind shear, and the last term is the log of the grid-cell area weight. Coefficients $b$ are calibrated against ERA5 over 1981–2010 (Fosu et al. 2024).

Output files are saved to:
- `{work_dir}/TCGI-CRH_{year}.nc`
- `{work_dir}/TCGI-SD_{year}.nc`


In [ ]:
# TCGI regression coefficients: [constant, vorticity, humidity, thermal, shear]
b_crh = [-24.1323, 2.5120, 0.0770, 0.0622, -0.1202]   # TCGI-CRH
b_sd  = [-18.3563, 2.4829, 0.0735, 0.0798, -0.1346]   # TCGI-SD

# Precompute the log area weight for each grid cell: ln(cos(lat) * dx * dy)
area     = np.log(np.cos(np.deg2rad(ds_out.lat)) * 2 * 2)
area_ext = np.tile(area, (12, len(ds_out.lon), 1)).transpose(0, 2, 1)  # shape: (12, lat, lon)

print('### Step 4: Calculating TCGI...')

for iyear in range(cal_start, cal_end + 1):
    bcp_ds = {
        var: xr.open_dataset(f"{work_dir}/unbiased/{var}_{iyear}_2deg_unbiased.nc")[var]
        for var in output_vars
    }

    # Construct the area weight DataArray aligned to this year's time axis
    offset = xr.DataArray(
        area_ext,
        dims=['time', 'lat', 'lon'],
        coords={'time': bcp_ds['avort']['time'], 'lat': ds_out.lat, 'lon': ds_out.lon,}
    )

    mu_crh = np.exp(
        b_crh[0]
        + b_crh[1] * bcp_ds['avort']
        + b_crh[2] * bcp_ds['crh']
        + b_crh[3] * bcp_ds['PI']
        + b_crh[4] * bcp_ds['ws']
        + offset
    )
    mu_sd = np.exp(
        b_sd[0]
        + b_sd[1] * bcp_ds['avort']
        + b_sd[2] * bcp_ds['sd']
        + b_sd[3] * bcp_ds['PI']
        + b_sd[4] * bcp_ds['ws']
        + offset
    )

    mu_crh.rename('TCGI').to_netcdf(
        f"{work_dir}/TCGI-CRH_{iyear}.nc",
        encoding={'TCGI': {'zlib': True, 'complevel': 5}}, mode='w'
    )
    mu_sd.rename('TCGI').to_netcdf(
        f"{work_dir}/TCGI-SD_{iyear}.nc",
        encoding={'TCGI': {'zlib': True, 'complevel': 5}}, mode='w'
    )
    print(f'  Saved TCGI for {iyear}')

print('### Step 4 complete.')